# 소비자원

In [0]:
%sql
CREATE OR REPLACE TABLE silver.kca_info.grocery_filtered AS
SELECT * FROM silver.kca_info.grocery
WHERE `대분류` IN ('신선식품', '양곡')

In [0]:
df2 = spark.table("silver.kca_info.grocery_filtered")
display(df2)

In [0]:
from pyspark.sql.functions import concat_ws

df2_with_unit = df2.withColumn("단위", concat_ws("", df2.goodTotalCnt, df2.goodTotalDivCode))

In [0]:
from pyspark.sql.functions import concat_ws, col
df2_with_unit = df2_with_unit.withColumn("지역", concat_ws("", col("시도명"), col("구군명")))

In [0]:
from pyspark.sql.functions import lit

df2_with_unit = df2_with_unit.withColumn("출처", lit("kca"))

In [0]:
# kca = df2_with_unit.select(
#     col("goodInspectDay").alias("날짜"),
#     col("출처"),
#     col("goodName").alias("재료명"),
#     col("단위"),
#     col("중분류").alias("카테고리"),
#     col("업태명"),
#     col("지역"),
#     col("goodprice").alias("가격"),
#     col("collect_time").alias("수집시간")
# )
# display(kca.distinct().orderBy("재료명", ascending=False))
# kca.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.kca_info.kca_standard")

from pyspark.sql.functions import col

kca = df2_with_unit.select(
    col("goodInspectDay").alias("날짜"),
    col("출처"),
    col("goodName").alias("재료명"),
    col("단위"),
    col("중분류").alias("카테고리"),
    col("업태명"),
    col("지역"),
    col("goodPrice").alias("가격"),
    col("collect_time").alias("수집시간")
)
display(kca.distinct().orderBy("재료명", ascending=False))
#kca.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.kca_info.kca_standard")

In [0]:
# # 포맷 설정 후 csv 저장 - 지역 컬럼 시도, 시군구로 분리 작업
# df_normalized = spark.table("silver.kca_info.kca_normalized")
# df_normalized.write.mode("overwrite").option("header", "true").csv("/Volumes/silver/kca_info/kca_normalized")

In [0]:
# df_kca_normalized_2 = spark.read.option("header", "true").csv("dbfs:/Volumes/silver/kca_info/kca_normalized/kca_normalized2.csv")
# display(df_kca_normalized_2)

In [0]:
from pyspark.sql import functions as F

# 시도 정규화 (풀네임 → 짧은 이름)
sido_normalize = {
    '서울특별시': '서울', '부산광역시': '부산', '대구광역시': '대구',
    '인천광역시': '인천', '광주광역시': '광주', '대전광역시': '대전',
    '울산광역시': '울산', '세종특별자치시': '세종',
    '경기도': '경기', '강원도': '강원', '강원특별자치도': '강원',
    '충청북도': '충북', '충청남도': '충남',
    '전라북도': '전북', '전북특별자치도': '전북', '전라남도': '전남',
    '경상북도': '경북', '경상남도': '경남',
    '제주특별자치도': '제주',
}

sido_pattern = '|'.join(sido_normalize.keys())
sido_normalize_expr = F.create_map([F.lit(k) for pair in sido_normalize.items() for k in pair])

# 시도 추출: 문자열 앞부분에서 광역자치단체명 추출
sido_raw = F.regexp_extract(F.col('지역'), rf'^({sido_pattern})', 1)

# 시군구 추출: 시도명 뒤에 오는 시/군/구 단위 추출
# '경기도수원시  권선' → '수원시', '서울특별시송파구' → '송파구'
sigungu_raw = F.regexp_extract(F.col('지역'), rf'^(?:{sido_pattern})([가-힣]+(?:시|군|구))', 1)


지역_idx = kca.columns.index('지역')
cols_before = kca.columns[:지역_idx]
cols_after  = kca.columns[지역_idx + 1:]

kca_df = kca.select(
    *cols_before,
    F.when(sido_raw != '', sido_normalize_expr[sido_raw])
     .alias('시도'),
    F.when((sigungu_raw != '') & ~sigungu_raw.isin(list(sido_normalize.keys())), sigungu_raw)
     .otherwise(F.lit(None)).alias('시군구'),
    *cols_after,
).drop('지역')

In [0]:
from pyspark.sql.functions import regexp_replace

kca_df = kca_df.withColumn("단위", regexp_replace("단위", "G", "g"))

In [0]:
display(kca_df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructType, StructField

# 65개 unique '재료명' → (세부 속성, 재료명 핵심 키워드) 매핑
mapping = {
    '1등급 다향 훈제오리(500g)':              ('1등급 다향 훈제오리',                     '훈제오리'),
    '96시간 숙성 현미(1kg)':                  ('96시간 숙성 현미',                        '현미'),
    'CJ 1등급 깨끗한 계란(10개)':             ('CJ 1등급 깨끗한 계란',                   '달걀'),
    'CJ 1등급 깨끗한 계란(15개)':             ('CJ 1등급 깨끗한 계란',                   '달걀'),
    'CJ 동물복지 유정란(15개)':                ('CJ 동물복지 유정란',                      '달걀'),
    '갈치(냉동, 100g)':                       ('갈치(냉동)',                              '갈치'),
    '갈치(생물, 100g)':                       ('갈치(생물)',                              '갈치'),
    '감자(껍질 있는 감자, 100g)':              ('감자(껍질 있는 감자)',                    '감자'),
    '고구마(껍질 있는 밤고구마, 100g)':         ('고구마(껍질 있는 밤고구마)',              '고구마'),
    '고구마(껍질 있는 호박고구마, 100g)':       ('고구마(껍질 있는 호박고구마)',            '고구마'),
    '고등어(생물, 300~500g)':                 ('고등어(생물)',                            '고등어'),
    '깻잎(100g)':                             ('깻잎',                                    '깻잎'),
    '노르웨이 생연어 필렛(250g)':              ('노르웨이 생연어 필렛',                    '연어'),
    '느타리버섯(100g)':                        ('느타리버섯',                              '느타리버섯'),
    '당근(흙당근, 100g)':                     ('당근(흙당근)',                            '당근'),
    '당진 해나루쌀 특등급(10kg)':              ('당진 해나루쌀 특등급',                    '쌀'),
    '당찬진미 특등급(10kg)':                  ('당찬진미 특등급',                         '쌀'),
    '대왕님표 여주 진상미(10kg)':              ('대왕님표 여주 진상미',                    '쌀'),
    '대왕님표 여주 진상미(4kg)':               ('대왕님표 여주 진상미',                    '쌀'),
    '대파(흙대파, 500~800g)':                 ('대파(흙대파)',                            '대파'),
    '대하(100g)':                             ('대하',                                    '대하'),
    '돼지고기 목살(100g)':                    ('돼지고기 목살',                           '돼지고기'),
    '돼지고기 삼겹살(100g)':                  ('돼지고기 삼겹살',                         '돼지고기'),
    '마늘(깐마늘, 100g)':                     ('마늘(깐마늘)',                            '마늘'),
    '목초를 먹고 자란 건강한 닭이 낳은 달걀(15개)': ('목초를 먹고 자란 건강한 닭이 낳은 달걀', '달걀'),
    '무(줄기 없는 무, 1.5kg)':                ('무(줄기 없는 무)',                        '무'),
    '바른고을 의성진쌀(10kg)':                 ('바른고을 의성진쌀',                       '쌀'),
    '배추(1.5~2kg)':                          ('배추',                                    '배추'),
    '부세(200~400g)':                         ('부세',                                    '부세'),
    '불릴필요없는 현미(2kg)':                  ('불릴필요없는 현미',                       '현미'),
    '쇠고기 등심(1+등급, 100g)':              ('쇠고기 등심(1+등급)',                    '쇠고기'),
    '쇠고기 등심(1등급, 100g)':               ('쇠고기 등심(1등급)',                     '쇠고기'),
    '쇠고기 불고기(1+등급, 100g)':            ('쇠고기 불고기(1+등급)',                  '쇠고기'),
    '쇠고기 불고기(1등급, 100g)':             ('쇠고기 불고기(1등급)',                   '쇠고기'),
    '스모크델리 훈제오리(500g)':               ('스모크델리 훈제오리',                     '훈제오리'),
    '시금치(250~400g)':                       ('시금치',                                  '시금치'),
    '애호박':                                 ('애호박',                                  '애호박'),
    '양배추':                                 ('양배추',                                  '양배추'),
    '양송이버섯(100g)':                        ('양송이버섯',                              '양송이버섯'),
    '양파(껍질 있는 망포장, 1.5kg)':           ('양파(껍질 있는 망포장)',                  '양파'),
    '오이(백다다기)':                          ('오이(백다다기)',                          '오이'),
    '오징어(냉동, 200~300g)':                 ('오징어(냉동)',                            '오징어'),
    '오징어(생물, 200~300g)':                 ('오징어(생물)',                            '오징어'),
    '의성마늘 훈제오리 슬라이스(400g)':         ('의성마늘 훈제오리 슬라이스',              '훈제오리'),
    '임금님표 이천쌀 특등급(10kg)':            ('임금님표 이천쌀 특등급',                  '쌀'),
    '임금님표 이천쌀(4kg)':                   ('임금님표 이천쌀',                         '쌀'),
    '적상추(100g)':                           ('적상추',                                  '적상추'),
    '정성식품 오징어젓(400g)':                 ('정성식품 오징어젓',                       '오징어젓'),
    '쪽파(흙쪽파)':                            ('쪽파(흙쪽파)',                            '쪽파'),
    '참조기(200~400g)':                       ('참조기',                                  '참조기'),
    '철원 오대쌀(10kg)':                       ('철원 오대쌀',                             '쌀'),
    '청정원 동물복지 청정유정란(15개)':          ('청정원 동물복지 청정유정란',              '달걀'),
    '청정원 자유방목 동물복지 유정란(10개)':     ('청정원 자유방목 동물복지 유정란',         '달걀'),
    '청정원 행복놀이터 동물복지 유정란(15개)':   ('청정원 행복놀이터 동물복지 유정란',       '달걀'),
    '풀무원 동물복지 목초란(10개)':             ('풀무원 동물복지 목초란',                  '달걀'),
    '풀무원 동물복지 목초란(15개)':             ('풀무원 동물복지 목초란',                  '달걀'),
    '풀무원 동물복지 유정란(10개)':             ('풀무원 동물복지 유정란',                  '달걀'),
    '풋고추(100g)':                           ('풋고추',                                  '풋고추'),
    '프리미엄 파타고니아 항공직송 생연어 필렛(500g)': ('프리미엄 파타고니아 항공직송 생연어 필렛', '연어'),
    '한성 오징어젓갈(150g)':                   ('한성 오징어젓갈',                         '오징어젓갈'),
    '항공직송 동원생연어(320g)':                ('항공직송 동원생연어',                     '연어'),
    '항공직송 동원생연어(350g)':                ('항공직송 동원생연어',                     '연어'),
    '햇살드리 수향미(10kg)':                   ('햇살드리 수향미',                         '쌀'),
    '허브를 담은 정다운 훈제오리(400g)':        ('허브를 담은 정다운 훈제오리',             '훈제오리'),
    '흰다리새우(100g)':                        ('흰다리새우',                              '새우'),
}

# 매핑 dict → 작은 lookup DataFrame으로 변환 (broadcast join 용)
mapping_rows = [(k, v[0], v[1]) for k, v in mapping.items()]
schema = StructType([
    StructField('_원재료명', StringType()),
    StructField('세부 속성',  StringType()),
    StructField('_재료명_신', StringType()),
])
mapping_df = spark.createDataFrame(mapping_rows, schema=schema)

# df는 이전 단계에서 이미 만들어져 있다고 가정
# broadcast join - mapping이 작아서 셔플 없이 처리됨
kca_df = (
    kca_df.join(F.broadcast(mapping_df), kca_df['재료명'] == mapping_df['_원재료명'], 'left')
      .drop('재료명', '_원재료명')
      .withColumnRenamed('_재료명_신', '재료명')
      # 컬럼 순서: '날짜', '출처', '재료명','세부 속성', '단위', '카테고리', '업태명', '시도', '시군구', '가격', '수집시간'
      .select('날짜', '출처', '재료명','세부 속성', '단위', '카테고리', '업태명', '시도', '시군구', '가격', '수집시간')
)

In [0]:
display(kca_df)

In [0]:
kca_df = kca_df.withColumnRenamed("세부 속성", "세부속성")

In [0]:
display(kca_df.select("단위").distinct())

In [0]:
from pyspark.sql.functions import regexp_extract, col, when, concat, lit

kca_df = kca_df.withColumn(
    "단위_수치", regexp_extract(col("단위"), r"(\d+)", 1).cast("int")
).withColumn(
    "단위_문자", regexp_extract(col("단위"), r"([a-zA-Z]+)", 1)
)

kca_df = kca_df.withColumn(
    "단위",
    when(
        (col("단위_문자") == "g") & (col("단위_수치") >= 1000),
        concat(
            ((col("단위_수치") / 1000).cast("int")).cast("string"),
            lit("kg")  
        )
    ).otherwise(col("단위"))
)

In [0]:
for code, kor in {
    'EA': '개',
    'DA': '단',
    'MA': '망',
    'MR': '마리',
    'PK': '포기'
}.items():
    kca_df = kca_df.withColumn(
        "단위",
        F.regexp_replace("단위", f"{code}$", kor)
    ).withColumn(
        "단위_문자",
        F.regexp_replace("단위_문자", f"^{code}$", kor)
    )

In [0]:
display(kca_df.limit(1000))

In [0]:
display(kca_df.select("단위").distinct())

In [0]:
kca_df.write.mode("overwrite").saveAsTable("silver.kca_info.kca_normalized2")

In [0]:
# display(df_kca_normalized_2)
# # display(df_kca_normalized_2.distinct().count()) # 642,087개
# # display(df_kca_normalized_2.count()) #2,143,880개